In [ ]:
import sys
sys.path.append("../")

import h5py
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scripts.resnet import ResNet1d
import torch


In [ ]:
beats_summary = pd.read_csv("../data/beats_summary_frame.csv")
beats_summary.head()

In [ ]:
# Read the raw configuration and model data, as well as
# the exam metadata and raw ECG signals.

from scripts.constants import (
    DATA_DIR,
    N_LEADS,
)
config = '../model/config.json'

# Instantiate the model using the config.json information.
with open(config, 'r') as f:
    config_dict = json.load(f)
model = ResNet1d(
    input_dim=(N_LEADS, config_dict['seq_length']),
    blocks_dim=list(zip(config_dict['net_filter_size'], config_dict['net_seq_lengh'])),
    n_classes=1,
    kernel_size=config_dict['kernel_size'],
    dropout_rate=config_dict['dropout_rate']
)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Retrieve the state dict, which has all the coefficients
state_dict = (torch.load('../model/model.pth',
              weights_only=False,
              map_location=device))

# Load the state dict and set the model to eval mode.
model.load_state_dict(state_dict['model'])
model.eval()

# Read in exam metadata and limit to file 16.
metadata = pd.read_csv(f'{DATA_DIR}/exams.csv')
metadata = metadata[metadata['trace_file'] == 'exams_part16.hdf5']

# Read in raw ECG data for file 16.
filename = "../data/exams_part16.hdf5"

with h5py.File(filename, "r") as f:
    print("Keys in the HDF5 file:", list(f.keys()))
    dataset = f['tracings']
    print("Dataset shape:", dataset.shape)
    print("Dataset dtype:", dataset.dtype)
    data_array = f['tracings'][()]
    exam_ids = f['exam_id'][()]

In [ ]:
# Brute force sort df to match order of exam_ids
metadata_sorted = []
for exam_id in exam_ids:
    metadata_sorted.append(metadata[metadata['exam_id'] == exam_id])
metadata = pd.concat(metadata_sorted)
metadata['subject'] = np.arange(metadata.shape[0])
metadata.head()

In [ ]:
# Get information on which subjects to retain
retain_subjects = beats_summary.groupby('subject')['retain_subject'].first().reset_index()
retain_subjects = retain_subjects[:20000]
retain_subjects = retain_subjects[retain_subjects['retain_subject'] == True]


In [ ]:
# Limit to 20k observations
data_array = data_array[:20000, :, :]

In [ ]:
keep_only_retain_subjects = True
if keep_only_retain_subjects:
    data_array = data_array[retain_subjects['subject'].values, :, :]
    metadata = metadata[metadata['subject'].isin(retain_subjects['subject'].values)]
    beats_summary = beats_summary[beats_summary['subject'].isin(retain_subjects['subject'].values)]

In [ ]:
deletion_start = -0.1
deletion_end = -0.05

data_array_del = data_array.copy()
areas = []
for subject in range(data_array.shape[0]):
    subject_number = retain_subjects.iloc[subject]['subject']
    area = 0
    for channel in range(data_array.shape[2]):
        peaks = beats_summary.loc[(beats_summary['subject'] == subject_number)
                                    & (beats_summary['channel'] == channel), 'peaks'].values[0]
        peaks = [int(item) for item in peaks.replace("[", "").replace("]", "").split()]
        if len(peaks) >= 2:
            avg_beat_length = (peaks[-1] - peaks[0]) / (len(peaks) - 1)
            deletion_starts = [int(i + (deletion_start * avg_beat_length)) for i in peaks]
            deletion_ends = [int(i + (deletion_end * avg_beat_length)) for i in peaks]
            for i in range(len(peaks)):
                if (deletion_starts[i] >= 0) and (deletion_ends[i] < 4096):
                    replace_val = (
                        np.linspace(data_array[subject, deletion_starts[i], channel],
                                    data_array[subject, deletion_ends[i], channel],
                                    deletion_ends[i] - deletion_starts[i])
                        )
                    data_array_del[subject, deletion_starts[i]:deletion_ends[i], channel] = (
                        replace_val
                        )
                    area += float(
                        np.sum(
                            np.abs(
                                data_array[subject, 
                                           deletion_starts[i]:deletion_ends[i], 
                                           channel] 
                                - replace_val)
                            )
                        )
    areas.append(area)


In [ ]:
n_total = data_array.shape[0]  # total number of predictions
batch_size = 10
n_batches = int(np.ceil(n_total / batch_size))

mse = []
pred_list = []
end = 0
for i in range(n_batches):
    if (i % 100 == 0):
        print(i, "of", n_batches)
    start = end
    end = min((i + 1) * batch_size, n_total)

    # Get the predictions
    model.zero_grad()
    y_pred = model(torch.tensor(data_array_del[start:end, :, :], dtype=torch.float).transpose(-1, -2))

    # Merge predictions back onto the metadata frame
    preds = pd.DataFrame({'exam_id': metadata['exam_id'][start:end],
                          'torch_pred': y_pred.detach().numpy().squeeze()})
    pred_list.append(preds)

preds = pd.concat(pred_list, axis=0, ignore_index=True)
compare = metadata.merge(preds, on='exam_id', how='inner')
mse.append(float(np.mean((compare['nn_predicted_age'] - compare['torch_pred'])**2)))


In [ ]:
mse

In [ ]:
out_frame = pd.merge(preds, metadata, on='exam_id')
out_frame['area_removed'] = areas
out_frame['start_pct'] = deletion_start
out_frame['end_pct'] = deletion_end